# Pi0 Hacking Notebook Part 6 — PyTorch Version

Same exploration as the JAX notebook, but using the PyTorch implementation for easier tracing.
The PyTorch model lets us hook into individual layers, set breakpoints, and inspect activations
without fighting `nn.scan` or the NNX bridge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import copy
import math
import time

import torch
import torch.nn.functional as F

# openpi imports
from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.policies import aloha_policy
from openpi.shared import download
from openpi import transforms as _transforms
from openpi.shared import normalize as _normalize
from openpi.models import model as _model

print("Imports OK")

In [ ]:
joint_labels = [
    "L joint 0 (waist)", "L joint 1 (shoulder)", "L joint 2 (elbow)",
    "L joint 3 (forearm)", "L joint 4 (wrist angle)", "L joint 5 (wrist rotate)",
    "L gripper",
    "R joint 0 (waist)", "R joint 1 (shoulder)", "R joint 2 (elbow)",
    "R joint 3 (forearm)", "R joint 4 (wrist angle)", "R joint 5 (wrist rotate)",
    "R gripper",
]

In [ ]:
def frame_to_obs(frame):
    """Convert a HuggingFace dataset frame to the observation dict
    that the policy expects (post-RepackTransform format)."""
    def to_chw(img):
        arr = np.array(img)  # PIL -> HWC (H, W, 3)
        return arr.transpose(2, 0, 1)  # -> CHW (3, H, W)
    
    return {
        "images": {
            "cam_high": to_chw(frame["observation.images.cam_high"]),
            "cam_left_wrist": to_chw(frame["observation.images.cam_left_wrist"]),
            "cam_right_wrist": to_chw(frame["observation.images.cam_right_wrist"]),
        },
        "state": np.array(frame["observation.state"]),
        "prompt": "uncap the pen",
    }

## 1. Load Dataset

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
print(f"Total rows: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"Episode indices: {sorted(set(ds['episode_index']))}")

In [ ]:
# Pick an episode and grab all its frames
EPISODE_IDX = 0

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)

In [ ]:
# Pick a frame in the middle of the episode
FRAME_IDX = 150
frame = episode[FRAME_IDX]

In [ ]:
# Display camera views
cam_names = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, cam_names):
    img = frame[f"observation.images.{name}"]
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle(f"Episode {EPISODE_IDX}, Frame {FRAME_IDX} (t={frame['timestamp']:.2f}s)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Load PyTorch Model

In [ ]:
config = _config.get_config("pi0_aloha_pen_uncap")

# Point to converted PyTorch checkpoint
checkpoint_dir = "data/robots/openpi/pi0_aloha_pen_uncap_pytorch"

print(f"Model type: {config.model.model_type}")
print(f"Action dim: {config.model.action_dim}")
print(f"Action horizon: {config.model.action_horizon}")

In [ ]:
# Create the trained policy — auto-detects PyTorch format
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print(f"Policy created successfully!")
print(f"PyTorch model: {policy._is_pytorch_model}")
print(f"Device: {policy._pytorch_device}")
print(f"Number of input transforms: {len(policy._input_transform.transforms)}")
print(f"Number of output transforms: {len(policy._output_transform.transforms)}")

In [ ]:
# Quick reference inference to make sure the model works
obs_test = frame_to_obs(frame)
result_test = policy.infer(obs_test)
print(f"Actions shape: {result_test['actions'].shape}")
print(f"Inference time: {result_test['policy_timing']['infer_ms']:.1f} ms")

## 3. Reference Inference with Fixed Noise

Generate fixed noise (using numpy so we don't need JAX at all),
run `policy.infer` as a black box, and save the result as our reference.

In [ ]:
# Fix noise for reproducibility (numpy, no JAX dependency)
rng = np.random.RandomState(42)
fixed_noise = rng.randn(50, 32).astype(np.float32)
print(f"fixed_noise: {fixed_noise.shape}, dtype={fixed_noise.dtype}")
print(f"  norm={np.linalg.norm(fixed_noise):.4f}, mean={fixed_noise.mean():.4f}")

# Reference result using policy.infer as a black box
obs_ref = frame_to_obs(frame)
result_ref = policy.infer(obs_ref, noise=fixed_noise)
print(f"\nReference actions: {result_ref['actions'].shape}")
print(f"  range: [{result_ref['actions'].min():.4f}, {result_ref['actions'].max():.4f}]")

## 4. Manual Reproduction — Input Transforms

Replicate what `policy.infer()` does step by step.

In [ ]:
device = torch.device(policy._pytorch_device)
pt_model = policy._model  # the PI0Pytorch instance
print(f"Device: {device}")
print(f"Model type: {type(pt_model).__name__}")

In [ ]:
# Step 1: Input transforms (same chain as JAX version)
obs_manual = frame_to_obs(frame)
inputs = policy._input_transform(obs_manual)

print("After input transforms:")
for k, v in inputs.items():
    if isinstance(v, dict):
        for k2, v2 in v.items():
            print(f"  {k}/{k2}: {v2.shape}, dtype={v2.dtype}")
    elif isinstance(v, np.ndarray):
        print(f"  {k}: {v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
# Step 2: Batch and convert to PyTorch tensors — each leaf [shape] -> torch [1, shape]
import jax  # just for tree.map utility, no JAX computation
inputs_pt = jax.tree.map(
    lambda x: torch.from_numpy(np.array(x)).to(device)[None, ...],
    inputs
)

print("Batched PyTorch tensors:")
for k, v in inputs_pt.items():
    if isinstance(v, dict):
        for k2, v2 in v.items():
            print(f"  {k}/{k2}: {v2.shape}, dtype={v2.dtype}, device={v2.device}")
    elif isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}, dtype={v.dtype}")

In [ ]:
# Step 3: Build Observation dataclass
# For torch uint8 images, from_dict converts to float32 and permutes to BCHW
observation = _model.Observation.from_dict(inputs_pt)

print("Observation:")
for name, img in observation.images.items():
    print(f"  image[{name}]: {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
for name, m in observation.image_masks.items():
    print(f"  mask[{name}]: {m.shape}, dtype={m.dtype}")
print(f"  state: {observation.state.shape}, dtype={observation.state.dtype}")
print(f"  tokenized_prompt: {observation.tokenized_prompt.shape}")
print(f"  tokenized_prompt_mask: {observation.tokenized_prompt_mask.shape}, valid={observation.tokenized_prompt_mask.sum()}")

## 5. Preprocess + Embed Prefix

Now we enter the model internals. The PyTorch model's `_preprocess_observation`
handles image resizing, then `embed_prefix` runs SigLIP on images and embeds language tokens.

In [ ]:
# Step 4: Preprocess observation (no augmentation at inference)
# This unpacks the Observation into lists that embed_prefix expects
images, img_masks, lang_tokens, lang_masks, state = pt_model._preprocess_observation(
    observation, train=False
)

print("Preprocessed observation:")
for i, (img, mask) in enumerate(zip(images, img_masks)):
    print(f"  image[{i}]: {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
    print(f"  mask[{i}]:  {mask.shape}, val={mask}")
print(f"  lang_tokens: {lang_tokens.shape}, dtype={lang_tokens.dtype}")
print(f"  lang_masks: {lang_masks.shape}, valid={lang_masks.sum()}")
print(f"  state: {state.shape}, dtype={state.dtype}")

In [ ]:
# Step 5: Embed prefix using model's embed_prefix method
with torch.no_grad():
    prefix_embs, prefix_pad_masks, prefix_att_masks = pt_model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks
    )

print(f"prefix_embs:      {prefix_embs.shape}, dtype={prefix_embs.dtype}")      # [1, 816, 2048]
print(f"prefix_pad_masks: {prefix_pad_masks.shape}, dtype={prefix_pad_masks.dtype}")  # [1, 816]
print(f"prefix_att_masks: {prefix_att_masks.shape}, dtype={prefix_att_masks.dtype}")  # [1, 816]

### 5b. Expand embed_prefix manually

Let's do what `embed_prefix` does, but step by step so we can inspect each piece.
This is much easier to trace than the JAX version!

In [ ]:
with torch.no_grad():
    embs_list = []
    pad_mask_list = []
    ar_mask_list = []

    # --- Encode each image through SigLIP ---
    # In PyTorch: paligemma.model.get_image_features
    # This is HuggingFace's PaliGemma vision tower (SigLIP So400m/14)
    for i, (img, img_mask) in enumerate(zip(images, img_masks)):
        img_emb = pt_model.paligemma_with_expert.embed_image(img)
        bsize, num_img_tokens = img_emb.shape[:2]

        print(f"SigLIP image {i}:")
        print(f"  input:  {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
        print(f"  output: {img_emb.shape}, dtype={img_emb.dtype}")

        embs_list.append(img_emb)
        pad_mask_list.append(img_mask[:, None].expand(bsize, num_img_tokens))
        ar_mask_list += [0] * num_img_tokens

    # --- Embed language tokens ---
    # PaliGemma's embedding table + scale by sqrt(dim)
    lang_emb = pt_model.paligemma_with_expert.embed_language_tokens(lang_tokens)
    lang_emb_dim = lang_emb.shape[-1]
    lang_emb = lang_emb * math.sqrt(lang_emb_dim)

    print(f"\nLanguage embedding:")
    print(f"  token IDs: {lang_tokens.shape}, non-padding: {lang_masks.sum()}")
    print(f"  embeddings: {lang_emb.shape}, dtype={lang_emb.dtype}")

    embs_list.append(lang_emb)
    pad_mask_list.append(lang_masks)
    ar_mask_list += [0] * lang_emb.shape[1]

    # --- Concatenate into full prefix sequence ---
    prefix_embs_manual = torch.cat(embs_list, dim=1)
    prefix_pad_masks_manual = torch.cat(pad_mask_list, dim=1)

    print(f"\nFull prefix:")
    print(f"  embs:      {prefix_embs_manual.shape}  (3x256 image + 48 language = 816)")
    print(f"  pad_masks: {prefix_pad_masks_manual.shape}  (valid: {prefix_pad_masks_manual.sum()})")

    # --- Compare against reference ---
    max_diff = (prefix_embs_manual - prefix_embs).abs().max().item()
    print(f"\nMax diff vs embed_prefix reference: {max_diff}")

In [ ]:
# pt_model.paligemma_with_expert.embed_image?
# pt_model.paligemma_with_expert.model.get_image_features

In [ ]:
# pt_model.paligemma_with_expert.paligemma.model.get_image_features

In [ ]:
lang_tokens

In [ ]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")

# # Decode the full sequence
# print(tokenizer.decode(lang_tokens[0]))

# # See individual tokens (including padding)
# tokens = tokenizer.convert_ids_to_tokens(lang_tokens[0].tolist())
# for i, (tok, mask) in enumerate(zip(tokens, lang_masks[0])):
#     print(f"  [{i:2d}] id={lang_tokens[0, i]:6d}  mask={mask.item()}  '{tok}'")

## 6. Prefix Forward Pass → KV Cache

Run the prefix tokens through the PaliGemma language model (18 layers) and cache the K, V.
This is the `[prefix_embs, None]` call — only expert 0 (PaliGemma) runs.

In [ ]:
from openpi.models_pytorch.pi0_pytorch import make_att_2d_masks

with torch.no_grad():
    # Build attention mask
    prefix_att_2d = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_positions = torch.cumsum(prefix_pad_masks, dim=1) - 1

    print(f"prefix_att_2d: {prefix_att_2d.shape}")  # [1, 816, 816]
    print(f"prefix_positions: {prefix_positions.shape}")  # [1, 816]

    # Convert to 4D attention mask (the format HuggingFace expects)
    prefix_att_4d = pt_model._prepare_attention_masks_4d(prefix_att_2d)
    print(f"prefix_att_4d: {prefix_att_4d.shape}")  # [1, 1, 816, 816]

    # Set eager attention (not SDPA) so we can hook into it
    pt_model.paligemma_with_expert.paligemma.language_model.config._attn_implementation = "eager"

    # Forward pass: [prefix_embs, None] — only PaliGemma runs, action expert gets None
    _, kv_cache = pt_model.paligemma_with_expert.forward(
        attention_mask=prefix_att_4d,
        position_ids=prefix_positions,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

print(f"\nKV cache type: {type(kv_cache).__name__}")
print(f"KV cache layers: {len(kv_cache)}")
# Each layer has (key, value) tensors
k0, v0 = kv_cache[0]
print(f"Layer 0 K: {k0.shape}, V: {v0.shape}")  # [1, num_kv_heads, 816, head_dim]

### Pre Softmax

In [ ]:
attn_logits = {}
attn_weights = {}
hooks = []

def make_deep_attn_hook(layer_idx):
    original_forward = None
    
    def patched_forward(self, *args, **kwargs):
        # Run the normal forward
        kwargs['output_attentions'] = True
        result = original_forward(*args, **kwargs)
        
        # Compute logits manually from Q, K
        # We can get them by hooking the softmax input instead
        return result
    
    # Simpler approach: hook the attention dropout or softmax
    def hook_fn(module, inp, out):
        # For GemmaAttention, we can compute logits from Q and K
        # But easier: just capture both scales
        if isinstance(out, tuple) and out[1] is not None:
            attn_weights[layer_idx] = out[1].detach().cpu()
    return hook_fn

# Actually, the cleanest approach: compute logits from Q and K directly
q_captures = {}
k_captures = {}

def make_q_hook(layer_idx):
    def hook_fn(module, inp, out):
        q_captures[layer_idx] = out.detach().cpu()
    return hook_fn

def make_k_hook(layer_idx):
    def hook_fn(module, inp, out):
        k_captures[layer_idx] = out.detach().cpu()
    return hook_fn

pali_layers = pt_model.paligemma_with_expert.paligemma.language_model.layers
for i, layer in enumerate(pali_layers):
    hooks.append(layer.self_attn.q_proj.register_forward_hook(make_q_hook(i)))
    hooks.append(layer.self_attn.k_proj.register_forward_hook(make_k_hook(i)))

with torch.no_grad():
    _, kv_cache = pt_model.paligemma_with_expert.forward(
        attention_mask=prefix_att_4d,
        position_ids=prefix_positions,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

for h in hooks:
    h.remove()

print(f"Captured Q and K for {len(q_captures)} layers")
print(f"Q shape: {q_captures[0].shape}")  # [1, 816, num_heads * head_dim]
print(f"K shape: {k_captures[0].shape}")  # [1, 816, num_kv_heads * head_dim]

In [ ]:
layer_idx = 12
head = 5

attn = pali_attn_patterns[layer_idx][0, head].float()

# log(softmax(logits)) = logits - logsumexp(logits)
# So log(attn) recovers the logits up to a per-row constant
# This preserves all the relative structure
log_attn = torch.log(attn + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im0 = axes[0].imshow(log_attn.numpy(), cmap='viridis',
                       vmin=log_attn[log_attn > -20].quantile(0.05).item(),
                       vmax=log_attn[log_attn > -20].quantile(0.95).item())
axes[0].set_title("Log attention (≈ logits up to row constant)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(attn.numpy(), cmap='viridis',
                       vmin=0, 
                       vmax=attn.quantile(0.99).item())
axes[1].set_title("Attention weights (clipped at 99th percentile)")
plt.colorbar(im1, ax=axes[1])

for ax in axes:
    for boundary in [256, 512, 768]:
        ax.axhline(y=boundary, color='cyan', linewidth=0.5, alpha=0.7)
        ax.axvline(x=boundary, color='cyan', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
layer_idx = 12
head_dim = 256
num_heads = 8

# Reshape Q to [batch, num_heads, seq, head_dim]
q = q_captures[layer_idx].float().reshape(1, 816, num_heads, head_dim).transpose(1, 2)
k = k_captures[layer_idx].float().reshape(1, 816, 1, head_dim).transpose(1, 2)

# Apply RoPE (need positions) — or skip for a rough look
# Without RoPE this is approximate but still shows structure
scaling = head_dim ** -0.5
logits = torch.matmul(q * scaling, k.transpose(-1, -2))  # [1, 8, 816, 816]

# Apply mask
mask_bool = prefix_pad_masks[0].cpu().bool()
logits[:, :, :, ~mask_bool] = float('-inf')

head = 5
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pre-softmax logits — more dynamic range visible
im0 = axes[0].imshow(logits[0, head].numpy(), cmap='viridis', 
                       # vmin=logits[0, head][mask_bool].quantile(0.05),
                       # vmax=logits[0, head][mask_bool].quantile(0.95)
                    )
axes[0].set_title("Pre-softmax logits (layer 12, head 5)")
plt.colorbar(im0, ax=axes[0])

# Post-softmax for comparison
probs = torch.softmax(logits, dim=-1)
im1 = axes[1].imshow(probs[0, head].numpy(), cmap='viridis')
axes[1].set_title("Post-softmax attention (layer 12, head 5)")
plt.colorbar(im1, ax=axes[1])

for ax in axes:
    for boundary in [256, 512, 768]:
        ax.axhline(y=boundary, color='cyan', linewidth=0.5, alpha=0.7)
        ax.axvline(x=boundary, color='cyan', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.show()

### Post SoftMax

In [ ]:
# === Capture PaliGemma attention patterns ===

pali_attn_patterns = {}
hooks = []

def make_attn_hook(layer_idx):
    def hook_fn(module, inp, output):
        # GemmaAttention.forward returns (attn_output, attn_weights, ...)
        if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
            # attn_weights shape: [batch, num_heads, seq_len, seq_len]
            pali_attn_patterns[layer_idx] = output[1].detach().cpu()
    return hook_fn

pali_layers = pt_model.paligemma_with_expert.paligemma.language_model.layers
for i, layer in enumerate(pali_layers):
    h = layer.self_attn.register_forward_hook(make_attn_hook(i))
    hooks.append(h)

print(f"Hooked {len(hooks)} attention layers")

# Run the prefix forward pass
with torch.no_grad():
    _, kv_cache = pt_model.paligemma_with_expert.forward(
        attention_mask=prefix_att_4d,
        position_ids=prefix_positions,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

# Clean up hooks
for h in hooks:
    h.remove()

# Check what we captured
print(f"\nCaptured {len(pali_attn_patterns)} layers")
for layer_idx in sorted(pali_attn_patterns.keys()):
    attn = pali_attn_patterns[layer_idx]
    print(f"  layer {layer_idx}: {attn.shape}")
    # Expecting [1, 8, 816, 816] — 8 query heads, 816 tokens

In [ ]:
plt.imshow(pali_attn_patterns[12].detach().cpu().float().numpy()[0,5])

In [ ]:
plt.imshow(pali_attn_patterns[12].detach().cpu().float().numpy()[0,5], 
           vmin=0, 
           vmax=pali_attn_patterns[12].detach().cpu().float().quantile(0.99).item())

In [ ]:
plt.imshow(pali_attn_patterns[12].detach().cpu().float().numpy()[0,0], 
           vmin=0, 
           vmax=pali_attn_patterns[12].detach().cpu().float().quantile(0.99).item())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

attn = pali_attn_patterns[12][0, 5].float().numpy()  # layer 12, head 5

# Full pattern
axes[0].imshow(attn, vmin=0, vmax=attn.max() * 0.5)
axes[0].set_title("Full attention pattern")
# Mark image/language boundaries
for boundary in [256, 512, 768]:
    axes[0].axhline(y=boundary, color='r', linewidth=0.5, alpha=0.7)
    axes[0].axvline(x=boundary, color='r', linewidth=0.5, alpha=0.7)
axes[0].set_xlabel("key position")
axes[0].set_ylabel("query position")

# Which tokens are attended TO most (column sums = "sink score")
sink_scores = attn.sum(axis=0)
axes[1].plot(sink_scores)
axes[1].set_title("Attention sink scores (column sums)")
axes[1].set_xlabel("key position")
for boundary in [256, 512, 768]:
    axes[1].axvline(x=boundary, color='r', linewidth=0.5, alpha=0.7)

# Which tokens ATTEND most broadly (row entropy)
# High entropy = attending broadly, low entropy = attending to few tokens
eps = 1e-10
entropy = -(attn * np.log(attn + eps)).sum(axis=1)
axes[2].plot(entropy)
axes[2].set_title("Row entropy (attention breadth)")
axes[2].set_xlabel("query position")
for boundary in [256, 512, 768]:
    axes[2].axvline(x=boundary, color='r', linewidth=0.5, alpha=0.7)

plt.suptitle("Layer 12, Head 5 — Red lines = image boundaries (256, 512, 768)")
plt.tight_layout()
plt.show()

In [ ]:
prefix_att_4d.shape

In [ ]:
plt.imshow(prefix_att_4d[0,0].detach().cpu().numpy())

- Right so not the diagonal attention mask we would see in autoregressive text generation
- Hmm i would like to visualize some attention patterns!

In [ ]:
pt_model.paligemma_with_expert

In [ ]:
# pt_model.paligemma_with_expert?

In [ ]:
state.shape

In [ ]:
state

In [ ]:
# x_t.shape

In [ ]:
# x_t

## 7. Flow Matching Denoising Loop

10 Euler steps from t=1 (noise) to t=0 (actions).
Each step embeds the suffix (state + noisy actions + timestep), runs the action expert
with the cached prefix KV, and takes an Euler step.

In [ ]:
num_steps = 10
dt = -1.0 / num_steps

# Start from fixed noise
noise_pt = torch.from_numpy(fixed_noise).to(device)[None, ...]  # [1, 50, 32]
x_t = noise_pt.clone()

# Collect trajectory for visualization
trajectory = [x_t[0].cpu().numpy().copy()]
velocities = []
t = 1.0

with torch.no_grad():
    for step in range(num_steps):
        timestep = torch.tensor([t], dtype=torch.float32, device=device)

        # === Embed suffix: state token + action/time tokens ===
        suffix_embs, suffix_pad_masks, suffix_att_masks, adarms_cond = pt_model.embed_suffix(
            state, x_t, timestep
        )
        if step == 0:
            print(f"suffix_embs: {suffix_embs.shape}")       # [1, 51, 1024] = 1 state + 50 action
            print(f"suffix_pad_masks: {suffix_pad_masks.shape}")
            print(f"suffix_att_masks: {suffix_att_masks.shape}")
            print(f"adarms_cond: {adarms_cond}")              # None for pi0

        # === Build attention mask ===
        # suffix queries attend to [prefix KV cache | suffix tokens]
        suffix_len = suffix_pad_masks.shape[1]
        batch_size = prefix_pad_masks.shape[0]
        prefix_len = prefix_pad_masks.shape[1]

        prefix_cross_mask = prefix_pad_masks[:, None, :].expand(batch_size, suffix_len, prefix_len)
        suffix_att_2d = make_att_2d_masks(suffix_pad_masks, suffix_att_masks)
        full_att_2d = torch.cat([prefix_cross_mask, suffix_att_2d], dim=2)

        if step == 0:
            print(f"full_att_2d: {full_att_2d.shape}")  # [1, 51, 816+51]

        # Suffix positions continue from where prefix left off
        prefix_offsets = torch.sum(prefix_pad_masks, dim=-1)[:, None]
        suffix_positions = prefix_offsets + torch.cumsum(suffix_pad_masks, dim=1) - 1

        # Convert to 4D mask
        full_att_4d = pt_model._prepare_attention_masks_4d(full_att_2d)

        # Set eager attention for action expert too
        pt_model.paligemma_with_expert.gemma_expert.model.config._attn_implementation = "eager"

        # === Forward pass: [None, suffix_embs] — action expert runs, PaliGemma skipped ===
        (prefix_out, suffix_out), _ = pt_model.paligemma_with_expert.forward(
            attention_mask=full_att_4d,
            position_ids=suffix_positions,
            past_key_values=kv_cache,
            inputs_embeds=[None, suffix_embs],
            use_cache=False,
            adarms_cond=[None, adarms_cond],
        )
        assert prefix_out is None

        # Extract velocity from last 50 tokens (action tokens, excluding state token)
        action_out = suffix_out[:, -pt_model.config.action_horizon:]
        v_t = pt_model.action_out_proj(action_out.to(torch.float32))

        if step == 0:
            print(f"\nv_t (velocity): {v_t.shape}")  # [1, 50, 32]

        # Euler step
        x_t = x_t + dt * v_t
        t = t + dt

        trajectory.append(x_t[0].cpu().numpy().copy())
        velocities.append(v_t[0].cpu().numpy().copy())
        print(f"  step {step}: t={t-dt:.1f}->{t:.1f}, |v_t|={v_t.norm():.2f}, |x_t|={x_t.norm():.2f}")

- How exaclty are models used together?
- What do attention patterns look like?


```
        (prefix_out, suffix_out), _ = pt_model.paligemma_with_expert.forward(
            attention_mask=full_att_4d,
            position_ids=suffix_positions,
            past_key_values=kv_cache,
            inputs_embeds=[None, suffix_embs],
            use_cache=False,
            adarms_cond=[None, adarms_cond],
        )
```

Hmm I think I might need to sketch this out, and then Chat with Claude some more.

In [ ]:
816+51

In [ ]:
full_att_4d.shape

In [ ]:
plt.imshow(full_att_4d[0,0].detach().cpu().numpy())

In [ ]:
suffix_out.shape

In [ ]:
action_out.shape

In [ ]:
pt_model.action_out_proj

In [ ]:
np.where(np.abs(np.diff(full_att_4d[0,0,0,:].detach().cpu().numpy())))

## 8. Compare Against Reference

In [ ]:
# Apply the same output transforms to our manually denoised actions
outputs_manual = {
    "state": np.asarray(inputs_pt["state"][0].cpu()),
    "actions": x_t[0].cpu().numpy(),
}
outputs_manual = policy._output_transform(outputs_manual)

max_diff = np.max(np.abs(result_ref["actions"] - outputs_manual["actions"]))
print(f"Max diff vs reference: {max_diff:.2e}")
print(f"All close (atol=1e-4): {np.allclose(result_ref['actions'], outputs_manual['actions'], atol=1e-4)}")
print(f"All close (atol=1e-3): {np.allclose(result_ref['actions'], outputs_manual['actions'], atol=1e-3)}")

## 9. Compare: Using denoise_step (cleaner API)

The PyTorch model has a nice self-contained `denoise_step` method.
Let's verify it matches our manual loop, then use it going forward.

In [ ]:
# Re-run using the model's denoise_step method
x_t_v2 = noise_pt.clone()
t_v2 = 1.0

with torch.no_grad():
    for step in range(num_steps):
        timestep = torch.tensor(t_v2, dtype=torch.float32, device=device).expand(1)
        v_t = pt_model.denoise_step(
            state, prefix_pad_masks, kv_cache, x_t_v2, timestep
        )
        x_t_v2 = x_t_v2 + dt * v_t
        t_v2 += dt

diff_v2 = (x_t_v2 - x_t).abs().max().item()
print(f"Max diff (manual loop vs denoise_step): {diff_v2:.2e}")

## 10. Plot Predicted vs Ground Truth Actions

In [ ]:
predicted_actions = outputs_manual["actions"]

In [ ]:
# Gather ground truth actions for the next action_horizon frames
action_horizon = predicted_actions.shape[0]
n_gt = min(action_horizon, len(episode) - FRAME_IDX)

gt_actions = np.array([episode[FRAME_IDX + t]["action"] for t in range(n_gt)])
print(f"Ground truth actions: {gt_actions.shape}")
print(f"Predicted actions: {predicted_actions.shape}")

In [ ]:
# Plot predicted vs ground truth for each joint
fig, axes = plt.subplots(7, 2, figsize=(16, 28), sharex=True)
time_steps = np.arange(n_gt)

for arm_idx, arm_name in enumerate(["Left", "Right"]):
    offset = arm_idx * 7
    for joint_idx in range(7):
        ax = axes[joint_idx, arm_idx]
        dim = offset + joint_idx
        
        ax.plot(time_steps, gt_actions[:n_gt, dim], 'b-', linewidth=2, label='Ground Truth', alpha=0.8)
        ax.plot(time_steps, predicted_actions[:n_gt, dim], 'r--', linewidth=2, label='Predicted', alpha=0.8)
        
        ax.set_ylabel(joint_labels[dim].replace(f"{arm_name[0]} ", ""), fontsize=9)
        ax.grid(True, alpha=0.3)
        if joint_idx == 0:
            ax.set_title(f"{arm_name} Arm", fontsize=12, fontweight='bold')
            ax.legend(fontsize=8)
        if joint_idx == 6:
            ax.set_xlabel("Timestep")

plt.suptitle(f"PyTorch Model: Predicted vs Ground Truth Actions (from frame {FRAME_IDX})", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11. Bonus: Hook Into Individual Layers

This is the main payoff of switching to PyTorch — we can register forward hooks
on any layer and capture activations as they flow through the transformer.

In [ ]:
# Register hooks on every layer of both experts
activations = {}
hooks = []

def make_hook(name):
    def hook_fn(module, inp, out):
        # Store detached copies to avoid memory issues
        if isinstance(out, torch.Tensor):
            activations[name] = out.detach().cpu()
        elif isinstance(out, tuple) and len(out) > 0:
            activations[name] = out[0].detach().cpu() if isinstance(out[0], torch.Tensor) else str(type(out[0]))
    return hook_fn

# Hook PaliGemma layers (prefix expert)
pali_layers = pt_model.paligemma_with_expert.paligemma.language_model.layers
for i, layer in enumerate(pali_layers):
    h = layer.register_forward_hook(make_hook(f"pali_layer_{i}"))
    hooks.append(h)

# Hook action expert layers
expert_layers = pt_model.paligemma_with_expert.gemma_expert.model.layers
for i, layer in enumerate(expert_layers):
    h = layer.register_forward_hook(make_hook(f"expert_layer_{i}"))
    hooks.append(h)

print(f"Registered {len(hooks)} hooks")
print(f"  PaliGemma layers: {len(pali_layers)}")
print(f"  Action expert layers: {len(expert_layers)}")

In [ ]:
# Run one denoising step and capture activations
activations.clear()

with torch.no_grad():
    timestep = torch.tensor([1.0], dtype=torch.float32, device=device)
    v_t_hooked = pt_model.denoise_step(
        state, prefix_pad_masks, kv_cache, noise_pt, timestep
    )

print("Captured activations:")
for name, act in sorted(activations.items()):
    if isinstance(act, torch.Tensor):
        print(f"  {name}: {act.shape}, dtype={act.dtype}")
    else:
        print(f"  {name}: {act}")

In [ ]:
# Clean up hooks when done
for h in hooks:
    h.remove()
hooks.clear()
print("Hooks removed")